# 步驟 1：資料載入與處理
讀取從 FinMind 快取下來的財務、價量資料，並轉換為寬表。

In [ ]:
import os, sys, glob, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm import tqdm

print("=" * 62)
print("  Taiwan ML Stock Strategy  v4  (Optimized)")
print("=" * 62)

# ─── Config ──────────────────────────────────────────────────────────────────
CACHE          = "finmind_cache"
TRAIN_MONTHS   = 36       # rolling train window
PURGE_MONTHS   = 1        # purge gap (data-leakage prevention)
STEP_MONTHS    = 3        # retrain every N months
TEST_START     = pd.Timestamp("2022-01-01")
TOP_K          = 20       # concentrated portfolio (was 30)
WEIGHT_TEMP    = 5.0      # softmax temperature for confidence weighting
FEE            = 1.425 / 1000 / 3
TAX            = 3 / 1000
STOP_LOSS      = 0.10
INIT_CASH      = 1_000_000

# ─── 1. Data loading ─────────────────────────────────────────────────────────
print("\n[1/7] Loading data...")

close = pd.read_pickle(os.path.join(CACHE, "close_wide.pkl"))
close.index = pd.to_datetime(close.index)
close.index.name = "date"
close.columns.name = "stock_id"

# Load high/low for ATR from price cache
print("  Loading OHLCV (for ATR & turnover)…")
high_frames, low_frames, money_frames = [], [], []
for f in tqdm(glob.glob(os.path.join(CACHE, "price/*.pkl")), desc="  OHLCV"):
    try:
        df = pd.read_pickle(f)
        df["date"] = pd.to_datetime(df["date"])
        for col, lst in [("max", high_frames), ("min", low_frames), ("Trading_money", money_frames)]:
            if col in df.columns:
                tmp = df[["date", "stock_id", col]].copy()
                tmp[col] = pd.to_numeric(tmp[col], errors="coerce")
                lst.append(tmp)
    except Exception:
        pass

def _wide(frames, col):
    df = pd.concat(frames, ignore_index=True)
    w = df.pivot_table(index="date", columns="stock_id", values=col)
    w.index = pd.to_datetime(w.index)
    w.index.name = "date"
    return w

high_wide  = _wide(high_frames,  "max")
low_wide   = _wide(low_frames,   "min")
money_wide = _wide(money_frames, "Trading_money")

# Revenue
print("  Loading revenue…")
rev_frames = []
for f in tqdm(glob.glob(os.path.join(CACHE, "revenue/*.pkl")), desc="  Revenue"):
    try:
        df = pd.read_pickle(f)
        df["date"] = pd.to_datetime(df["date"])
        rev_frames.append(df[["date", "stock_id", "revenue"]])
    except Exception:
        pass
rev_wide = (
    pd.concat(rev_frames, ignore_index=True)
    .pivot_table(index="date", columns="stock_id", values="revenue")
)
rev_wide.index = pd.to_datetime(rev_wide.index)
rev_wide.index.name = "date"

# Institution
print("  Loading institution…")
fg_frames, tr_frames = [], []
for f in tqdm(glob.glob(os.path.join(CACHE, "institution/*.pkl")), desc="  Inst"):
    try:
        df = pd.read_pickle(f)
        df["date"] = pd.to_datetime(df["date"])
        df["net"] = df["buy"] - df["sell"]
        fg_frames.append(df[df["name"] == "Foreign_Investor"][["date", "stock_id", "net"]])
        tr_frames.append(df[df["name"] == "Investment_Trust"][["date", "stock_id", "net"]])
    except Exception:
        pass

foreign_net = _wide(fg_frames, "net")
trust_net   = _wide(tr_frames, "net")

vol_wide = money_wide  # proxy for volume in money terms

print(f"  close={close.shape}  high={high_wide.shape}  rev={rev_wide.shape}")

# ─── 2. Factor engineering ────────────────────────────────────────────────────